# 08. Structural mean shifts

Detect conservative piecewise-constant mean shifts separately inside the historical and the modern statistical regime, so the known 1995 splice is never a candidate economic break.

**Reads**

- `outputs/tables/structural_breaks.csv`
- `outputs/tables/structural_break_bic_ladder.csv`
- `outputs/tables/structural_break_sensitivity.csv`
- `outputs/tables/structural_break_stability.csv`
- `data/processed/fiscal_balances_1977_2025.csv`

**Writes**

- Nothing. All four break tables are persisted by the pipeline.

**Method reference:** `METHODOLOGY.md` section 9

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

%matplotlib inline

from portugal_fiscal_balance.analysis import figures

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. Why the model is deliberately modest

Fewer than fifty annual observations, split across two statistical regimes, do
not support a flexible change-point model. The specification is therefore:

- a piecewise-constant mean, with at most two breaks per regime;
- a minimum segment length of five years;
- exact dynamic-programming minimisation of within-segment squared error;
- BIC model selection, which may select zero breaks;
- separate estimation for 1977-1994 and 1995-2025.

In [ ]:
breaks = pd.read_csv(TABLES / 'structural_breaks.csv')
display(breaks.round(3))

## 2. Fitted segments

Each chart shows one balance series with the selected segment means drawn on top
and the break years marked. A flat line across the whole regime means BIC
selected no break at all.

In [ ]:
panel = pd.read_csv(PROCESSED / 'fiscal_balances_1977_2025.csv')
figure = figures.structural_break_segments(
    panel, breaks, sector='general_government', regime='1995-2025_modern'
)

In [ ]:
figure = figures.structural_break_segments(
    panel, breaks, sector='social_security_funds', regime='1995-2025_modern'
)

In [ ]:
figure = figures.structural_break_segments(
    panel, breaks, sector='central_government', regime='1977-1994_historical'
)

## 3. How firm are those dates?

With eighteen or thirty-one observations per regime, a single selected date should
not be read as determined. Three guards are reported.

**The BIC ladder.** Publishing only the selected break count hides how close the
alternatives were. The ladder scores every admissible count so the margin is
visible.

In [ ]:
ladder = pd.read_csv(TABLES / 'structural_break_bic_ladder.csv')
display(
    ladder.pivot_table(
        index=['regime', 'sector'], columns='n_breaks', values='delta_bic_vs_best'
    ).round(2)
)

**The sensitivity grid.** Neither tuning parameter is estimated from the data,
so a date that survives only one of their values is a property of that choice
rather than of the series. Detection is re-run over all twelve combinations of a
minimum segment length in 4, 5, 6, 7 and a maximum of 1, 2 or 3 breaks.

In [ ]:
sensitivity = pd.read_csv(TABLES / 'structural_break_sensitivity.csv')
print('specifications per series:', len(sensitivity) // sensitivity.groupby(['regime', 'sector']).ngroups)
display(
    sensitivity.pivot_table(
        index=['regime', 'sector'], columns=['max_breaks', 'min_segment'], values='n_breaks'
    )
)

**The stability summary.** `modal_break_years_share` is the fraction of grid
cells returning exactly the modal set of dates. It is the quantity that decides
whether a date can be stated as detected or only as a candidate.

In [ ]:
stability = pd.read_csv(TABLES / 'structural_break_stability.csv')
display(stability.round(3))

## Interpretation limits

1. Break dates are **candidates, not findings**. The preferred specification
   identifies shifts around the years listed; the share columns say how much of
   the specification grid agrees. This notebook attaches no historical cause to
   any of them.
2. Detection is run **within** each regime. A shift at the 1995 boundary is
   unidentifiable here by construction, which is the intent.
3. A **five-year minimum segment** means shifts near the end of the sample cannot
   be detected yet, and the sensitivity grid shows how the detected dates move
   when that length is changed.
4. Selecting a mean shift does **not** imply the underlying series is
   piecewise-constant; it is the best fit within a restricted model class.
5. **BIC differences are not tests.** A small margin means the data do not
   distinguish the alternatives, not that the selected model is rejected.

---

[Previous: 07. Balance persistence and sign transitions](07_persistence.ipynb) | [Next: 09. Social Security Funds mechanisms](09_social_security_mechanisms.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```